# FireAI: “Қай аймақ қашан қауіпті?” — уақыт бойынша дұрыс Train/Test (2020–2024 train, 2025 test)

Бұл ноутбук **өрт тәуекелін** (fire risk) кеңістік-уақыт ұяшықтары бойынша болжайды.

Негізгі айырмашылықтар:
- ✅ **Random split жоқ** → **уақыт бойынша бөлу** (2020–2024 train, 2025 test)
- ✅ **Leakage жоқ** → label арқылы “ауа райын” қолдан жасап, модельді алдамау
- ✅ **0/1 dataset дұрыс** → (cell, month) комбинацияларының бәрін құрып, өрт жоқ жерлерге **0** қоямыз
- ✅ Нәтиже: әр ай үшін **қай ұяшық/аймақ қауіпті** екенін ықтималдықпен шығарамыз

> Ескерту: Егер сенде ауа райы/NDVI сияқты деректер болса, оларды `FEATURES_EXTRA` арқылы қосуға болады.

In [1]:
# 1) КІТАПХАНАЛАР
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
)

import folium
from folium.plugins import HeatMap

import warnings
warnings.filterwarnings("ignore")

## 2) ДЕРЕКТІ ОҚУ

Сен өзіңдегі FIRMS (NASA) CSV файлына жолды қой:
- `latitude`, `longitude`, `acq_date` бағандары болуы керек.

In [2]:
# 2) CSV PATH (ӨЗІҢНІҢ ФАЙЛ ЖОЛЫҢДЫ ҚОЙ)
CSV_PATH = r"C:\Users\Murat\Downloads\DL_FIRE_SV-C2_719890\fire_archive_SV-C2_719890.csv"

# Қазақстанға жуық bbox (қалауыңша өзгерте аласың)
LAT_MIN, LAT_MAX = 41.0, 55.0
LON_MIN, LON_MAX = 46.0, 87.0

df_raw = pd.read_csv(CSV_PATH)
need_cols = {"latitude", "longitude", "acq_date"}
missing = need_cols - set(df_raw.columns)
if missing:
    raise ValueError(f"CSV ішінде міндетті бағандар жоқ: {missing}. Бар бағандар: {list(df_raw.columns)[:30]}")

df = df_raw.loc[:, ["latitude", "longitude", "acq_date"]].copy()
df["acq_date"] = pd.to_datetime(df["acq_date"], errors="coerce")
df = df.dropna(subset=["acq_date", "latitude", "longitude"])

# Қазақстан bbox фильтрі
df = df[
    (df["latitude"].between(LAT_MIN, LAT_MAX)) &
    (df["longitude"].between(LON_MIN, LON_MAX))
].reset_index(drop=True)

print("Rows after KZ bbox filter:", len(df))
df.head()

Rows after KZ bbox filter: 162609


,latitude,longitude,acq_date
0,52.08270,76.85133,2023-01-01
1,51.25482,75.65141,2023-01-01
2,51.25476,75.65083,2023-01-01
3,50.01774,78.73595,2023-01-01
4,50.01782,78.73632,2023-01-01


## 3) КЕҢІСТІК (GRID) + УАҚЫТ (MONTH BIN)

Біз “аймақ” ұғымын тор (grid cell) арқылы жасаймыз:
- `Δlat`, `Δlon` — тор қадамы (0.1° ≈ 11 км)
- Уақыт — ай (month)

Содан кейін әр `(cell, month)` үшін label жасаймыз:
- `y=1` егер сол айда сол ұяшықта ≥1 өрт болса
- `y=0` әйтпесе

In [3]:
# 3) GRID + TIME BIN ПАРАМЕТРЛЕРІ
DELTA_LAT = 0.1
DELTA_LON = 0.1

# time_bin = ай
df["month"] = df["acq_date"].dt.to_period("M")  # мыс: 2025-07

# grid индекстер
df["i"] = np.floor((df["latitude"] - LAT_MIN) / DELTA_LAT).astype(int)
df["j"] = np.floor((df["longitude"] - LON_MIN) / DELTA_LON).astype(int)

# бір ұяшық идентификаторы
df["cell_id"] = df["i"].astype(str) + "_" + df["j"].astype(str)

# (cell, month) бойынша өрт санын агрегаттау
agg = (
    df.groupby(["cell_id", "i", "j", "month"])
      .size()
      .reset_index(name="fire_count")
)

# label
agg["y"] = (agg["fire_count"] > 0).astype(int)

print("Aggregated rows (only where fires exist):", len(agg))
agg.head()

Aggregated rows (only where fires exist): 20416


,cell_id,i,j,month,fire_count,y
0,0_221,0,221,2024-08,1,1
1,0_221,0,221,2024-11,2,1
2,0_222,0,222,2025-09,2,1
3,0_222,0,222,2025-10,1,1
4,0_225,0,225,2023-03,2,1


## 4) 0-КЛАСС (ӨРТ ЖОҚ) ҚҰРУ — ЕҢ МАҢЫЗДЫ ҚАДАМ

Модельге **өрт жоқ** ұяшық-ай комбинацияларын да беру керек.

Ол үшін біз:
- барлық cell тізімін (bbox ішіндегі grid)
- барлық ай тізімін (деректегі min→max)
құрып, толық тор жасаймыз.

Сосын өрт болғандарын `y=1`, қалғанын `y=0` қыламыз.

In [4]:
# 4) FULL GRID (cell x month) құру

# айлар диапазоны
all_months = pd.period_range(df["month"].min(), df["month"].max(), freq="M")

# bbox ішіндегі барлық grid ұяшықтары
i_max = int(np.floor((LAT_MAX - LAT_MIN) / DELTA_LAT))
j_max = int(np.floor((LON_MAX - LON_MIN) / DELTA_LON))

all_cells = []
for ii in range(0, i_max + 1):
    for jj in range(0, j_max + 1):
        cell_id = f"{ii}_{jj}"
        all_cells.append((cell_id, ii, jj))

cells_df = pd.DataFrame(all_cells, columns=["cell_id", "i", "j"])

# cell x month толық кросс
full = (
    cells_df.assign(key=1)
    .merge(pd.DataFrame({"month": all_months, "key": 1}), on="key", how="outer")
    .drop(columns=["key"])
)

# өрт санағын join
data = full.merge(agg[["cell_id", "month", "fire_count"]], on=["cell_id", "month"], how="left")
data["fire_count"] = data["fire_count"].fillna(0).astype(int)
data["y"] = (data["fire_count"] > 0).astype(int)

print("Full dataset rows (cell x month):", len(data))
print("Positive rate (y=1):", data["y"].mean())
data.head()

Full dataset rows (cell x month): 2086236
Positive rate (y=1): 0.009786045298806079


,cell_id,i,j,month,fire_count,y
0,0_0,0,0,2023-01,0,0
1,0_0,0,0,2023-02,0,0
2,0_0,0,0,2023-03,0,0
3,0_0,0,0,2023-04,0,0
4,0_0,0,0,2023-05,0,0


## 5) ФИЧАЛАР (FEATURE ENGINEERING)

Минималды, бірақ шынайы жұмыс істейтін фичалар:
- cell ортасының lat/lon
- ай (month) маусымдылық үшін
- синус/косинус (айдың циклдік табиғаты)
- лагтар (өткен айдағы өрт саны): `lag1`, `lag3`, `lag12`

> Егер сенде ауа райы/NDVI деректері болса — дәл осы жерге қосасың.

In [5]:
# 5) FEATURES

# cell ортасының координаталары
data["lat_center"] = LAT_MIN + (data["i"] + 0.5) * DELTA_LAT
data["lon_center"] = LON_MIN + (data["j"] + 0.5) * DELTA_LON

# ай нөмірі
data["month_num"] = data["month"].dt.month.astype(int)

# циклдік кодтау
data["month_sin"] = np.sin(2*np.pi*data["month_num"]/12)
data["month_cos"] = np.cos(2*np.pi*data["month_num"]/12)

# лаг фичалар: әр cell ішінде уақыт бойынша shift
data = data.sort_values(["cell_id", "month"]).reset_index(drop=True)

for k in [1, 3, 12]:
    data[f"fire_lag{k}"] = data.groupby("cell_id")["fire_count"].shift(k).fillna(0)

# Қосымша: moving average
data["fire_ma3"] = (
    data.groupby("cell_id")["fire_count"]
        .rolling(window=3, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
)

# final feature list
FEATURES = [
    "lat_center", "lon_center",
    "month_sin", "month_cos",
    "fire_lag1", "fire_lag3", "fire_lag12",
    "fire_ma3"
]

TARGET = "y"

data[FEATURES + [TARGET, "month"]].head()

,lat_center,lon_center,month_sin,month_cos,fire_lag1,fire_lag3,fire_lag12,fire_ma3,y,month
0,41.05,46.05,0.500000,8.660254e-01,0.0,0.0,0.0,0.0,0,2023-01
1,41.05,46.05,0.866025,5.000000e-01,0.0,0.0,0.0,0.0,0,2023-02
2,41.05,46.05,1.000000,6.123234e-17,0.0,0.0,0.0,0.0,0,2023-03
3,41.05,46.05,0.866025,-5.000000e-01,0.0,0.0,0.0,0.0,0,2023-04
4,41.05,46.05,0.500000,-8.660254e-01,0.0,0.0,0.0,0.0,0,2023-05


## 6) ДҰРЫС TRAIN/TEST БӨЛУ (УАҚЫТ БОЙЫНША)

- Train: 2020–2024 (яғни 2025-01-01-ге дейін)
- Test: 2025 (2025-01-01…2025-12-31)

Нақтысы:
- `train = month < 2025-01`
- `test = 2025-01 ≤ month < 2026-01`

Бұл — “болашақты” шынайы тексеру.

In [6]:
# 6) TIME SPLIT
train = data[data["month"] < pd.Period("2025-01", freq="M")].copy()
test  = data[(data["month"] >= pd.Period("2025-01", freq="M")) & (data["month"] < pd.Period("2026-01", freq="M"))].copy()

print("Train months:", train["month"].min(), "→", train["month"].max(), "rows:", len(train), "pos_rate:", train["y"].mean())
print("Test  months:", test["month"].min(),  "→", test["month"].max(),  "rows:", len(test),  "pos_rate:", test["y"].mean())

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

Train months: 2023-01 → 2024-12 rows: 1390824 pos_rate: 0.008442477265275836
Test  months: 2025-01 → 2025-12 rows: 695412 pos_rate: 0.012473181365866566


## 7) МОДЕЛЬ (RandomForest) + БАҒАЛАУ

Мақсат — ықтималдық беру:
\[
p_{c,\tau} = P(y_{c,\tau}=1|X_{c,\tau})
\]

Сондықтан `predict_proba` қолданамыз.

In [7]:
# 7) MODEL
model = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

proba_test = model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)

metrics = {
    "accuracy": accuracy_score(y_test, pred_test),
    "precision": precision_score(y_test, pred_test, zero_division=0),
    "recall": recall_score(y_test, pred_test, zero_division=0),
    "f1": f1_score(y_test, pred_test, zero_division=0),
}

print("Metrics (2025 test):")
for k,v in metrics.items():
    print(f"  {k:9s}: {v:.4f}")

# ROC AUC тек екі класс бар болса мағынасы болады (және y_test-та 0/1 екеуі де болса)
if y_test.nunique() == 2:
    print("  roc_auc  :", roc_auc_score(y_test, proba_test).round(4))

print("\nClassification report:")
print(classification_report(y_test, pred_test, digits=4))

Metrics (2025 test):
  accuracy : 0.9942
  precision: 0.7118
  recall   : 0.8943
  f1       : 0.7927
  roc_auc  : 0.9984

Classification report:
              precision    recall  f1-score   support

           0     0.9987    0.9954    0.9970    686738
           1     0.7118    0.8943    0.7927      8674

    accuracy                         0.9942    695412
   macro avg     0.8552    0.9449    0.8949    695412
weighted avg     0.9951    0.9942    0.9945    695412



## 8) “ҚАЙ АЙМАҚ ҚАШАН ҚАУІПТІ?” — НӘТИЖЕ КЕСТЕСІ

Әр ай үшін ең қауіпті **Top-N grid ұяшықтарын** шығарамыз (2025 test ішінде).

`risk_score = p_{c,τ}` — ұяшық деңгейіндегі қауіптілік ықтималдығы.

In [8]:
# 8) TOP CELLS PER MONTH
test_out = test.copy()
test_out["risk_proba"] = proba_test

TOP_N = 10

top_by_month = (
    test_out.sort_values(["month", "risk_proba"], ascending=[True, False])
            .groupby("month")
            .head(TOP_N)
            .loc[:, ["month", "cell_id", "lat_center", "lon_center", "risk_proba", "fire_count"]]
            .reset_index(drop=True)
)

top_by_month.head(20)

,month,cell_id,lat_center,lon_center,risk_proba,fire_count
0,2025-01,93_111,50.35,57.15,0.994839,29
1,2025-01,90_270,50.05,73.05,0.989559,26
2,2025-01,58_289,46.85,74.95,0.979670,21
3,2025-01,39_81,44.95,54.15,0.975957,13
4,2025-01,48_72,45.85,53.25,0.950824,25
5,2025-01,73_114,48.35,57.45,0.940882,9
6,2025-01,51_190,46.15,65.05,0.933784,9
7,2025-01,102_296,51.25,75.65,0.927924,15
8,2025-01,72_113,48.25,57.35,0.907878,5
9,2025-01,68_114,47.85,57.45,0.898343,8


### Айға қарай қауіптілік индексі (аймақтық жиынтық)

Егер аймақ \(R\) — көптеген ұяшық болса, онда:

- **Күтілетін өрт ұяшық саны (score):**
\[
Score(R,\tau)=\sum_{c\in R} p_{c,\tau}
\]
- **Кемінде бір өрт болуы ықтималдығы:**
\[
P(\ge1\ \text{fire in }R,\tau)=1-\prod_{c\in R}(1-p_{c,\tau})
\]

Төменде бүкіл Қазақстан bbox үшін “айлық қауіптілік индексін” есептейміз (2025 test).

In [9]:
# 8b) Kazakhstan-wide monthly risk indices
monthly = (
    test_out.groupby("month")
            .apply(lambda g: pd.Series({
                "Score_sum_p": float(g["risk_proba"].sum()),
                "P_at_least_1_fire": float(1 - np.prod(1 - g["risk_proba"].clip(0, 1)))
            }))
            .reset_index()
)

monthly

,month,Score_sum_p,P_at_least_1_fire
0,2025-01,57.402884,1.0
1,2025-02,119.297200,1.0
2,2025-03,1365.919384,1.0
3,2025-04,2702.759075,1.0
4,2025-05,1857.256157,1.0
5,2025-06,1234.041985,1.0
6,2025-07,797.853816,1.0
7,2025-08,748.060433,1.0
8,2025-09,664.514495,1.0
9,2025-10,1301.574594,1.0


## 9) КАРТА: ТАҢДАЛҒАН АЙ ҮШІН ТӘУЕКЕЛ HEATMAP

Мысалы: 2025-07 (шілде). `MONTH_TO_VIEW`-ды өзгерте аласың.

In [11]:
# 9) MAP FOR A SELECTED MONTH
MONTH_TO_VIEW = "2025-07"  # мыс: "2025-04", "2025-08" ...

m_period = pd.Period(MONTH_TO_VIEW, freq="M")
g = test_out[test_out["month"] == m_period].copy()

if len(g) == 0:
    raise ValueError(f"{MONTH_TO_VIEW} айы test ішінде табылмады. Дерек ай диапазонын тексер.")

kz_map = folium.Map(location=[48.0, 68.0], zoom_start=5, tiles="CartoDB positron")

# heatmap: lat, lon, weight=risk_proba
heat = g.loc[:, ["lat_center", "lon_center", "risk_proba"]].values.tolist()
HeatMap(heat, radius=12, blur=18, max_zoom=1).add_to(kz_map)

title_html = f'''
     <h3 align="center" style="font-size:16px"><b>FireAI Risk Heatmap — {MONTH_TO_VIEW}</b></h3>
'''
kz_map.get_root().html.add_child(folium.Element(title_html))

kz_map

## 10) ҚОЛДАНУ НҰСҚАУЛЫҒЫ (ҚЫСҚА)

1) `CSV_PATH`-ты өз файлыңа қой  
2) `DELTA_LAT/DELTA_LON`-ды қалауыңа қарай таңда (0.05–0.25 аралығы жиі қолданылады)  
3) Дерек 2020–2025 аралығын қамтыса, split автоматты дұрыс істейді  
4) Нәтижені:
   - `top_by_month` арқылы “қай ұяшық/аймақ қауіпті” деп шығарасың
   - `monthly` арқылы “қай ай/маусым қауіпті” деп шығарасың
   - картада `MONTH_TO_VIEW` арқылы көрсетесің

